# Индексация корпуса нормативных актов Банка России

Считает матрицы эмбеддингов для поискового слоя. Перед запуском подключить
датасет `ru-fin-retrieval` и включить ускоритель **GPU T4 x2**.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip install -q -U sentence-transformers


In [ ]:
!cp /kaggle/input/ru-fin-retrieval/*.py /kaggle/working/
%cd /kaggle/working


## Пробный прогон

Двадцать актов и одна лёгкая модель: проверяем, что датасет виден, код
импортируется и модель скачивается. Занимает пару минут.


In [ ]:
!python index_kaggle.py \
    --acts /kaggle/input/ru-fin-retrieval/acts.jsonl.gz \
    --limit-acts 20 --headline-models e5-small \
    --ablation-configs --out /kaggle/working/test


## Полный прогон

Пять моделей на базовой нарезке плюс абляции по нарезке на e5-small.
Посчитанное не пересчитывается, поэтому при обрыве ячейку можно просто
запустить заново.


In [ ]:
!python index_kaggle.py --acts /kaggle/input/ru-fin-retrieval/acts.jsonl.gz


## Что получилось


In [ ]:
import json
rep = json.load(open('/kaggle/working/embeddings/report.json'))
for r in rep:
    if 'error' in r:
        print(f"ОШИБКА {r['config']} / {r['model']}: {r['error']}")
    else:
        print(f"{r['config']:<12} {r['model']:<12} {r['chunks']:>7} чанков "
              f"dim={r['dim']:<5} {r['per_second']:>6} чанк/с {r['size_mb']:>7} МБ")
print(f"\nскачать обратно: {sum(r.get('size_mb', 0) for r in rep)/1024:.1f} ГБ")


Скачать папку `/kaggle/working/embeddings` и положить в проект
как `data/embeddings`.
